# 离散耦合水印：真实末步容量 canary
此 notebook 只运行两个预定开发源，捕获真实末步概率并计算可写容量上限。
不会执行水印优化、训练、确认集或多对鲁棒性实验。
修复版 v2：兼容旧组件路径和 Transformer 的 norm_type 缺失；原始缓存与权重保持不变。
18 项 CPU 检查通过，含小型完整管线加载/采样检查；完整预训练模型的 GPU 重跑待验证。
选择 GPU runtime 后 Run all；需要手动完成 Google Drive mount。
结果写入独立时间目录，失败和未运行样本均保留。通过只能说明接口运行/容量未排除，不能说明水印有效。


In [ ]:
import base64, io, zipfile
from pathlib import Path
task_dir = Path('/content/ceg_discrete_pair_v1')
task_dir.mkdir(exist_ok=True)
payload = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as archive:
    archive.extractall(task_dir)
print((task_dir/'protocol.json').read_text())


In [ ]:
import sys
sys.path.insert(0, str(task_dir))
from process_log import run_logged
run_logged([sys.executable, '-u', '-m', 'pip', 'install', '--quiet',
               'diffusers==0.39.0', 'transformers==4.57.6',
               'accelerate', 'sentencepiece', 'safetensors', 'pytest'])
run_logged([sys.executable, '-u', '-m', 'pytest', '-q', 'test_coupling.py',
            'test_capacity_interface.py', 'test_process_log.py'], cwd=task_dir)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import datetime, os
drive_root = Path('/content/drive/MyDrive/CEG-WM/discrete-pair-v1')
drive_root.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(drive_root/'model-cache')
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
run_dir = drive_root/'runs'/('capacity-canary-'+stamp)
print('Output:', run_dir)
log_path = run_dir.with_suffix('.log')
print('Log:', log_path)
try:
    run_logged([sys.executable, '-u', 'capacity_runner.py', '--count', '2', '--output', str(run_dir)],
               cwd=task_dir, log_path=log_path)
finally:
    status_path = run_dir/'status.json'
    if status_path.exists():
        print('Saved status:', status_path.read_text())


In [ ]:
status_path = run_dir/'status.json'
if status_path.exists():
    status = __import__('json').loads(status_path.read_text())
    print(__import__('json').dumps({k:status.get(k) for k in
          ['capacity_summary', 'stop_reason', 'wall_seconds', 'statuses']}, indent=2, ensure_ascii=False))
print('Stop here. Two-source canary is not full development evidence; review the records before expansion.')
